In [1]:
# Chapitre 1
from elasticsearch import Elasticsearch
import json

In [2]:
es = Elasticsearch("http://elasticsearch:9200")
# print(es.info())

In [3]:
res = es.search(
     index="films",
     query={"match_all": {}}
)

print(json.dumps(res["hits"]["hits"], indent=2))

[
  {
    "_index": "films",
    "_id": "1",
    "_score": 1.0,
    "_source": {
      "title": "Le Fabuleux Destin d'Am\u00e9lie Poulain",
      "summary": "Une jeune femme timide transforme la vie des gens autour d'elle \u00e0 Montmartre.",
      "genres": [
        "comedie",
        "romance"
      ],
      "director": "Jean-Pierre Jeunet",
      "year": 2001,
      "rating": 8.3
    }
  },
  {
    "_index": "films",
    "_id": "2",
    "_score": 1.0,
    "_source": {
      "title": "Le Seigneur des Anneaux : La Communaut\u00e9 de l'Anneau",
      "summary": "Un hobbit h\u00e9rite d'un anneau et part dans une qu\u00eate fantastique pour le d\u00e9truire.",
      "genres": [
        "fantasy",
        "aventure"
      ],
      "director": "Peter Jackson",
      "year": 2001,
      "rating": 8.8
    }
  },
  {
    "_index": "films",
    "_id": "3",
    "_score": 1.0,
    "_source": {
      "title": "Interstellar",
      "summary": "Des astronautes traversent l'espace et le temps pour

In [28]:
res = es.search(
    index="shakespeare",
    query={
        "term": {
            "text_entry": "love"
        }
    }
)

print(json.dumps(res["hits"]["hits"], indent=2))

[
  {
    "_index": "shakespeare",
    "_id": "5jMqXZ0BQKws95t1cnnd",
    "_score": 2.9574926,
    "_source": {
      "play_name": "Hamlet",
      "speaker": "HAMLET",
      "text_entry": "Love makes us weak",
      "year": 1603
    }
  },
  {
    "_index": "shakespeare",
    "_id": "5zMqXZ0BQKws95t1cnnm",
    "_score": 2.5466106,
    "_source": {
      "play_name": "Othello",
      "speaker": "OTHELLO",
      "text_entry": "I know this love is true",
      "year": 1604
    }
  }
]


In [4]:
"""
match et term peuvent parfois donner les mêmes résultats si le mot recherché correspond exactement 
au token stocké dans l’index inversé après analyse du texte.
"""

res = es.indices.analyze(
    index="shakespeare",
    body={
        "field": "text_entry",
        "text": "love"
    }
)

print(res)

{'tokens': [{'token': 'love', 'start_offset': 0, 'end_offset': 4, 'type': '<ALPHANUM>', 'position': 0}]}


In [37]:
res = es.search(
    index="shakespeare",
    query={
        "bool": {
            "must": [
                {"match": {"text_entry": "be"}}
            ],
            "filter": [
                {"term": {"play_name": "Hamlet"}}
            ]
        }
    }
)

print(json.dumps(res["hits"]["hits"], indent=2))

[]


In [40]:
# Oui mais on peut mettre hamlet et ça marche ( token inversé ) 
res = es.indices.analyze(
    index="shakespeare",
    body={
        "field": "play_name",
        "text": "Hamlet"
    }
)

print(res)

{'tokens': [{'token': 'hamlet', 'start_offset': 0, 'end_offset': 6, 'type': '<ALPHANUM>', 'position': 0}]}


In [42]:
# si on prend keyword ça marche, car là on demande la valeur exact 
res = es.search(
    index="shakespeare",
    query={
        "bool": {
            "must": [
                {"match": {"text_entry": "be"}}
            ],
            "filter": [
                {"term": {"play_name.keyword": "Hamlet"}}
            ]
        }
    }
)

print(json.dumps(res["hits"]["hits"], indent=2))

[
  {
    "_index": "shakespeare",
    "_id": "1jMnXZ0BQKws95t163mS",
    "_score": 2.4415183,
    "_source": {
      "play_name": "Hamlet",
      "speaker": "HAMLET",
      "text_entry": "To be or not to be",
      "year": 1603
    }
  },
  {
    "_index": "shakespeare",
    "_id": "3jMoXZ0BQKws95t1qHl5",
    "_score": 2.4415183,
    "_source": {
      "play_name": "Hamlet",
      "speaker": "HAMLET",
      "text_entry": "To be or not to be",
      "year": 1603
    }
  },
  {
    "_index": "shakespeare",
    "_id": "1",
    "_score": 2.0485835,
    "_source": {
      "play_name": "Hamlet",
      "speaker": "HAMLET",
      "text_entry": "To be, or not to be: that is the question."
    }
  },
  {
    "_index": "shakespeare",
    "_id": "zjNrWZ0BQKws95t1knkI",
    "_score": 1.7684224,
    "_source": {
      "play_name": "Hamlet",
      "speaker": "HAMLET",
      "text_entry": "I am and I will be"
    }
  }
]


In [44]:
mapping = es.indices.get_mapping(index="shakespeare")

print(mapping)

{'shakespeare': {'mappings': {'properties': {'play_name': {'type': 'text', 'fields': {'keyword': {'type': 'keyword', 'ignore_above': 256}}}, 'speaker': {'type': 'text', 'fields': {'keyword': {'type': 'keyword', 'ignore_above': 256}}}, 'text_entry': {'type': 'text', 'fields': {'keyword': {'type': 'keyword', 'ignore_above': 256}}}, 'year': {'type': 'long'}}}}}
